# Keystroke verification with generative models: experiments

CMU benchmark (51 users x 8 sessions x 50 typings of `.tie5Roanl`). One typing = one data point.
Features: 21 log timings (11 H + 10 DD); UD is dropped because UD = DD - H exactly. 31 raw features
are used only to reproduce published results (see `tests/`).

**Protocol.** Per user: train on typings 1-200 (sessions 1-4), genuine test 201-400 (sessions 5-8),
impostor test = first 5 typings of each of the other 50 users. Score: higher = more genuine.
EER per user -> mean, SD, bootstrap 95% CI over users. Paired comparisons vs **Scaled Manhattan | 21 log**.

**Tuning** only on validation (fit 1-160, validate 161-200, impostors = others' typings 6-10). Then refit
on 1-200 with the chosen settings and evaluate on test **once**. Population models never see impostor test
typings, and are also evaluated **open-set** (3 folds of 17 users; held-out fold = impostors only).
Every fit goes through `utils.check_no_leak`.

All writing to disk happens in the **saving cell** below; experiment cells only call
`start_run`, `save_run`, `save_model`, `save_figure`.

In [ ]:
import os, sys, json, re, io, time, shutil, platform, subprocess, warnings, contextlib
from pathlib import Path
from datetime import datetime
os.environ.setdefault("OMP_NUM_THREADS", "1")  # avoids the sklearn KMeans/MKL leak warning on Windows

REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

import numpy as np, pandas as pd, matplotlib, matplotlib.pyplot as plt, torch
from IPython.display import display
from src import utils as U, models as M, plots as P, experiments as E, flow as F
from src.evaluation import evaluate, evaluate_openset, ModelSpec

SEED = 0
np.random.seed(SEED); torch.manual_seed(SEED)
pd.set_option("display.width", 200, "display.max_columns", 30, "display.precision", 4)

df = U.load_data()                       # MD5-checked; stops with instructions if missing
X, NAMES = U.make_features(df, "log21")  # main feature set
RAW21, RAW_NAMES = U.make_features(df, "raw21")
print(df.shape, "users:", df.subject.nunique(), "| log21 features:", X.shape[1])
T0 = time.time()

## Saving cell (the only cell that writes files)

In [ ]:
# ============================ SAVING LOGIC ============================
OUT = REPO / "outputs"
REPORT_FIG = OUT / "report_figures"
INDEX = OUT / "runs_index.csv"
OUT.mkdir(exist_ok=True); REPORT_FIG.mkdir(exist_ok=True)


def _slug(s):
    return re.sub(r"[^A-Za-z0-9]+", "_", str(s)).strip("_")


def _jsonable(o):
    if isinstance(o, dict):
        return {str(k): _jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_jsonable(v) for v in o]
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, np.generic):
        return o.item()
    if isinstance(o, pd.DataFrame):
        return o.to_dict(orient="records")
    if isinstance(o, float) and not np.isfinite(o):
        return None
    if isinstance(o, (str, int, float, bool)) or o is None:
        return o
    return str(o)


def _write_json(path, obj):
    path.write_text(json.dumps(_jsonable(obj), indent=2))


def _env_info():
    try:
        commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
        dirty = bool(subprocess.check_output(["git", "status", "--porcelain", "--", "src", "notebooks"],
                                             cwd=REPO, text=True).strip())
    except Exception as e:
        commit, dirty = f"unavailable ({e})", None
    import scipy, sklearn
    return {"git_commit": commit, "git_dirty_src_or_notebooks": dirty, "python": platform.python_version(),
            "platform": platform.platform(), "numpy": np.__version__, "pandas": pd.__version__,
            "scipy": scipy.__version__, "scikit-learn": sklearn.__version__, "torch": torch.__version__,
            "matplotlib": matplotlib.__version__, "conda_env": os.environ.get("CONDA_DEFAULT_ENV")}


class _Tee(io.TextIOBase):
    def __init__(self, *streams):
        self.streams = streams

    def write(self, s):
        for st in self.streams:
            st.write(s)
        return len(s)

    def flush(self):
        for st in self.streams:
            st.flush()


class Run:
    def __init__(self, name, path, config):
        self.name, self.path, self.config = name, path, config
        self.id = path.name

    @contextlib.contextmanager
    def capture(self):
        # Tee printed output into log.txt and append any warnings.
        with open(self.path / "log.txt", "a", encoding="utf-8") as f, \
                warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("default")
            old = sys.stdout
            sys.stdout = _Tee(old, f)
            t = time.time()
            try:
                yield self
            finally:
                sys.stdout = old
                for w in caught:
                    f.write(f"WARNING {w.category.__name__}: {w.message} ({w.filename}:{w.lineno})\n")
                f.write(f"[elapsed {time.time() - t:.1f}s]\n")


def start_run(name, config):
    # Create outputs/<NAME><NNN>_<timestamp>/ (never overwrite) and write config.json + env.json BEFORE training.
    base = _slug(name)
    nums = [int(m.group(1)) for p in OUT.iterdir()
            if (m := re.fullmatch(re.escape(base) + r"(\d{3})_\d{8}-\d{6}", p.name))]
    nnn = max(nums, default=0) + 1
    path = OUT / f"{base}{nnn:03d}_{datetime.now():%Y%m%d-%H%M%S}"
    path.mkdir(parents=False, exist_ok=False)
    (path / "visualizations").mkdir()
    (path / "best_model").mkdir()
    _write_json(path / "config.json", {"run": path.name, "created": datetime.now().isoformat(), **config})
    _write_json(path / "env.json", _env_info())
    print(f"[run] {path.relative_to(REPO)}")
    return Run(name, path, config)


def save_figure(run, fig, name, report=False):
    p = run.path / "visualizations" / f"{_slug(name)}.png"
    fig.savefig(p, bbox_inches="tight")
    if report:
        fig.savefig(REPORT_FIG / f"{_slug(name)}.png", bbox_inches="tight")
    display(fig)
    plt.close(fig)
    return p


def save_model(run, models=None, best_config=None, grid_table=None, extra=None):
    # best_model/: best_config.json (+ grid results), per-user parameters (npz), sklearn joblib, flow state_dicts.
    import joblib
    bm = run.path / "best_model"
    cfg = dict(best_config or {})
    if grid_table is not None:
        grid_table.to_csv(bm / "grid_results.csv", index=False)
        cfg["grid_results"] = grid_table
    _write_json(bm / "best_config.json", cfg)
    if not models:
        return bm
    flows = {u: m.get_state() for u, m in models.items() if hasattr(m, "get_state")}
    if flows:
        torch.save(flows, bm / "flow_per_user.pt")
        pre = F.pretrained_models()
        if pre and any(m.pretrained is not None for m in models.values()):
            torch.save(pre, bm / "flow_pretrained.pt")
    sk = {u: {"gmm": m.gmm_, "std_mean": m.std_.mean_, "std_scale": m.std_.scale_}
          for u, m in models.items() if isinstance(m, M.GMM)}
    if sk:
        joblib.dump(sk, bm / "gmm_per_user.joblib")
    params = {f"user{u:02d}": m.get_params() for u, m in models.items() if not hasattr(m, "get_state")}
    if params:
        flat = {f"{u}__{k}": np.asarray(v) for u, d in params.items() for k, v in d.items()}
        np.savez_compressed(bm / "params_per_user.npz", **flat)
    if extra:
        _write_json(bm / "extra.json", extra)
    return bm


def save_run(run, per_user=None, metrics=None, scores=None, tables=None, index=None):
    # metrics.json, per_user.csv, scores.npz, extra tables; append a row to outputs/runs_index.csv.
    metrics = dict(metrics or {})
    _write_json(run.path / "metrics.json", metrics)
    if per_user is not None:
        per_user.to_csv(run.path / "per_user.csv", index=False)
    if scores:
        arrs = {}
        for k, (g, i) in scores.items():
            key = "_".join(map(str, k)) if isinstance(k, tuple) else str(k)
            arrs[f"genuine_{key}"], arrs[f"impostor_{key}"] = g, i
        np.savez_compressed(run.path / "scores.npz", **arrs)
    for name, t in (tables or {}).items():
        t.to_csv(run.path / f"{_slug(name)}.csv", index=False)
    c = run.config
    row = {"run_id": run.id, "timestamp": datetime.now().isoformat(timespec="seconds"),
           "model": c.get("model"), "key_settings": json.dumps(_jsonable(c.get("key_settings", {}))),
           "features": c.get("features"), "split": c.get("split_short", c.get("protocol")),
           "n_train": c.get("n_train"), "sampling": c.get("sampling"),
           "mean_eer": metrics.get("mean_eer"), "ci95_low": metrics.get("ci95_low"),
           "ci95_high": metrics.get("ci95_high"), "test_ll_median": metrics.get("test_ll_median")}
    row.update(index or {})
    pd.DataFrame([row]).to_csv(INDEX, mode="a", header=not INDEX.exists(), index=False)
    return run.path

## 0. Training-data overview

In [ ]:
run = start_run("data_overview", {
    "model": "none: data figures + overall Gaussian (pooled, and mixture of per-user Ledoit-Wolf Gaussians)",
    "features": "raw21 vs log21", "protocol": "descriptive", "split_short": "train(1-200)",
    "split": "training typings 1-200 of all users only", "n_train": 200, "sampling": "first", "seed": SEED})
with run.capture():
    user_lw = {u: M.LedoitWolfGaussian().fit(X[U.rows(u, 1, U.TRAIN_END)]) for u in range(U.N_USERS)}
    tr = U.rows(np.arange(U.N_USERS), 1, U.TRAIN_END)
    skew = lambda A: pd.DataFrame(A[tr]).skew().mean()
    stats = {"mean_skew_raw21": skew(RAW21), "mean_skew_log21": skew(X)}
    print(stats)
save_figure(run, P.raw_vs_log(RAW21, X, RAW_NAMES), "raw_vs_log_histograms", report=True)
save_figure(run, P.within_between(X, NAMES), "within_vs_between_user_spread", report=True)
save_figure(run, P.population_samples(X, user_lw, NAMES), "overall_gaussian_samples_vs_pooled", report=True)
save_figure(run, P.user_samples(X, user_lw, NAMES, users=[0, 20, 40],
                                title="Ledoit-Wolf Gaussian p(x|u): real vs samples"), "user_gaussian_samples")
save_model(run, user_lw, {"note": "per-user Ledoit-Wolf Gaussians on typings 1-200, used only for figures"})
save_run(run, metrics=stats)

## 1 + 2. Tuning on validation, then the official protocol (test once)

One run per model family: grid -> lowest mean validation EER -> refit on 1-200 -> single test evaluation.
For the flow, early stopping on validation picks the epoch count; the refit uses the median best epoch.

In [ ]:
TUNED, OFFICIAL, VAL_RUNS = {}, {}, {}
BASE = E.BASELINE
for fam, (make, g) in E.FAMILIES.items():
    pop = fam in E.POPULATION_FAMILIES
    run = start_run(f"official_{fam}", {
        "model": fam, "hyperparameter_grid": g, "selection": "lowest mean validation EER",
        "features": "log21 (11 H + 10 DD, log seconds)", "protocol": "official (Killourhy & Maxion)",
        "split_short": "val->test",
        "split": {"tuning": "fit typings 1-160, genuine 161-200, impostors = others' typings 6-10",
                  "test": "refit 1-200, genuine 201-400, impostors = others' typings 1-5 (250)"},
        "population": ("others' typings 11-200 (val) / 6-200 (test)" if pop and not fam.startswith("RealNVP")
                       else "pretraining: users outside the target's fold, typings 11-200 (val) / 6-200 (test)"
                       if pop else None),
        "n_train": 200, "sampling": "first", "seed": SEED})
    with run.capture():
        # Experiment 1: grid search on validation (sessions 5-8 are never touched here).
        best, table = E.tune_family(X, fam)
        # Refit on typings 1-200 with the chosen settings ...
        spec = E.final_spec(fam, best, table)
        print("chosen:", spec.label())
        # ... and evaluate on test exactly once (experiment 2).
        res = evaluate(X, spec, "test", keep_models=True)
        s = res.summary()
        print({k: round(v, 4) for k, v in s.items() if isinstance(v, float)})
    run.config["key_settings"] = best   # recorded in runs_index.csv (config.json was written before tuning)
    TUNED[fam], OFFICIAL[fam] = (best, table, spec), res

    # Save the selected model and the figures for this family.
    save_model(run, res.models, {"family": fam, "chosen": best, "final_spec": spec.label()}, table)
    if fam != BASE:
        save_figure(run, P.per_user_bars(res.eers, OFFICIAL[BASE].eers, fam, BASE + " | 21 log"), "per_user_eer")
    save_figure(run, P.det_curves({fam: res, BASE: OFFICIAL[BASE]}), "det_curve")
    if getattr(next(iter(res.models.values())), "is_density", False):
        save_figure(run, P.user_samples(X, res.models, NAMES, users=[0, 20, 40],
                                        title=f"{fam} p(x|u): real training typings vs samples"), "user_samples")
    if fam.startswith("RealNVP"):
        vr = evaluate(X, make(**best), "val", keep_models=True)  # validation run, for learning curves only
        save_figure(run, P.learning_curves({u: m.history_ for u, m in vr.models.items()},
                                           f"{fam}: learning curves (validation, early stopping)"),
                    "learning_curves", report=fam == "RealNVP")
    save_run(run, res.per_user, {**s, "chosen": best, "val_mean_eer": float(table.val_mean_eer[0])}, res.scores)

### Official-protocol table (paired vs Scaled Manhattan | 21 log)

In [ ]:
def paired_table(results, base_key):
    # One row per model: mean/SD/CI of the per-user EER, test log-likelihood, and a paired
    # comparison with the baseline on the same 51 users (negative diff = better than baseline).
    base = results[base_key].eers
    out = []
    for fam, r in results.items():
        s = r.summary()
        row = {"model": fam, "mean_eer": s["mean_eer"], "sd": s["sd_eer"], "ci95": f"[{s['ci95_low']:.4f}, {s['ci95_high']:.4f}]",
               "test_ll_median": s.get("test_ll_median", np.nan), "test_ll_min": s.get("test_ll_min", np.nan)}
        if fam != base_key:
            p = U.paired_test(r.eers, base)
            row.update(diff_vs_base=p["mean_diff"], diff_ci95=f"[{p['diff_ci95_low']:+.4f}, {p['diff_ci95_high']:+.4f}]",
                       wilcoxon_p=p["wilcoxon_p"], better_worse=f"{p['n_better']}/{p['n_worse']}")
        out.append(row)
    return pd.DataFrame(out).sort_values("mean_eer").reset_index(drop=True)

official_table = paired_table(OFFICIAL, BASE)
official_table.insert(1, "val_eer", official_table.model.map(lambda f: TUNED[f][1].val_mean_eer[0]))
official_table.insert(2, "chosen", official_table.model.map(lambda f: json.dumps(TUNED[f][0])))
run = start_run("table_official", {"model": "all", "features": "log21", "protocol": "official", "split_short": "test",
                                   "baseline": BASE + " | 21 log", "n_train": 200, "sampling": "first", "seed": SEED,
                                   "paired_tests": "bootstrap CI of per-user EER difference + Wilcoxon signed-rank"})
with run.capture():
    print(official_table.to_string())
wide = pd.DataFrame({f: r.eers for f, r in OFFICIAL.items()}).assign(user=U.users_of(df))
save_figure(run, P.det_curves(OFFICIAL, "DET, official protocol (log21)"), "det_official_all", report=True)
save_run(run, wide, {"table": official_table}, tables={"official_table": official_table},
         index={"mean_eer": official_table.loc[0, "mean_eer"], "model": "table (best: " + official_table.loc[0, "model"] + ")"})
display(official_table)

## 2b. Open-set evaluation of population models (3 folds of 17 users)

In [ ]:
OPEN = {}
for fam in [BASE, *E.POPULATION_FAMILIES]:
    best, table, spec = TUNED[fam]
    run = start_run(f"openset_{fam}", {
        "model": fam, "key_settings": best, "features": "log21", "protocol": "open-set", "split_short": "openset",
        "split": "folds of 17 users; held-out fold = impostors only (their typings 1-5); "
                 "population = non-held-out users' typings 1-200 (never the held-out fold); "
                 "per-user EER = mean over the 2 folds where the user is enrolled",
        "folds": [f.tolist() for f in U.openset_folds()], "n_train": 200, "sampling": "first", "seed": SEED})
    with run.capture():
        # Same hyperparameters as the official run; only the impostors and population change.
        res = evaluate_openset(X, spec)
        print(spec.label(), {k: round(v, 4) for k, v in res.summary().items() if isinstance(v, float)})
    OPEN[fam] = res
    save_model(run, None, {"family": fam, "chosen": best, "note": "hyperparameters from validation; models not kept (3 folds)"})
    save_run(run, res.per_user, res.summary(), res.scores)

openset_table = paired_table(OPEN, BASE)
openset_table["official_mean_eer"] = openset_table.model.map(lambda f: OFFICIAL[f].eers.mean())
run = start_run("table_openset", {"model": "population models", "features": "log21", "protocol": "open-set",
                                  "split_short": "openset", "baseline": BASE + " | 21 log (open-set impostors)",
                                  "n_train": 200, "sampling": "first", "seed": SEED})
with run.capture():
    print(openset_table.to_string())
save_run(run, pd.DataFrame({f: r.eers for f, r in OPEN.items()}).assign(user=U.users_of(df)),
         {"table": openset_table}, tables={"openset_table": openset_table},
         index={"mean_eer": openset_table.loc[0, "mean_eer"], "model": "table (best: " + openset_table.loc[0, "model"] + ")"})
display(openset_table)

## 3. Enrollment size

`first n` = one sitting (realistic). `stratified` = n/4 random typings per session 1-4, 5 seeds,
the same draws for all models (diagnostic: amount of data vs session drift). Hyperparameters are
the ones tuned on validation at n=160; test is always sessions 5-8 vs others' typings 1-5.

In [ ]:
ENROLL = []
for fam, (best, table, spec) in TUNED.items():
    run = start_run(f"enroll_{fam}", {
        "model": fam, "key_settings": best, "features": "log21", "protocol": "official test, varying enrollment",
        "split_short": "test", "n_train": list(E.ENROLL_N), "sampling": {"first": "typings 1..n",
        "stratified": "n/4 random per session 1-4, seeds " + str(list(E.STRAT_SEEDS))}, "seed": SEED})
    with run.capture():
        t = E.enrollment(X, spec).assign(model=fam)
        piv = t.groupby(["sampling", "n"])["eer"].mean().unstack(0)
        print(piv.round(4).to_string())
    ENROLL.append(t)
    save_model(run, None, {"family": fam, "chosen": best})
    save_run(run, t, {"mean_eer_by_n": piv, "mean_eer": float(piv.loc[200, "first"])},
             index={"sampling": "first+stratified", "n_train": "8..200"})
ENROLL = pd.concat(ENROLL, ignore_index=True)
enroll_table = ENROLL.groupby(["model", "sampling", "n"])["eer"].mean().unstack("n")
run = start_run("table_enrollment", {"model": "all", "features": "log21", "protocol": "enrollment size",
                                     "split_short": "test", "n_train": list(E.ENROLL_N), "sampling": "first+stratified", "seed": SEED})
with run.capture():
    print(enroll_table.round(4).to_string())
save_figure(run, P.enrollment_plot(ENROLL, "Enrollment size (log21): first-n vs stratified"), "enrollment_size", report=True)
save_run(run, ENROLL, {"table": enroll_table.reset_index()}, tables={"enrollment_table": enroll_table.reset_index()})
display(enroll_table.round(4))

## 4. Template adaptation

Enroll on the first n0 typings of session 1. Sessions 2-4 are presented chronologically as logins,
interleaved with 50 impostor attempts (others' typings 11-200, never test typings). Accepted typings are
added to the template; the threshold is the EER point on validation (session-1 typings n0+1..50 vs
others' typings 6-10), re-chosen after each refit. Test: sessions 5-8. `oracle` = template with all
genuine sessions 2-4 typings (perfect labels).

In [ ]:
ADAPT_FAMS = [BASE, "Laplace", "Gaussian diag", "Gaussian Ledoit-Wolf", "Gaussian Bayes/MAP", "Student-t", "GMM"]
ADAPT = []
for n0 in (12, 24):
    for fam in ADAPT_FAMS:
        best, table, spec = TUNED[fam]
        run = start_run(f"adapt_n{n0}_{fam}", {
            "model": fam, "key_settings": best, "features": "log21", "protocol": "template adaptation",
            "split_short": "adapt->test", "n_train": n0, "sampling": "first (session 1)",
            "adaptation": {"stream": "sessions 2-4 (150 genuine) + 50 impostor attempts (others' 11-200)",
                           "threshold": "EER point on validation (session 1 rest vs others' 6-10)", "refit_every": 10},
            "seed": SEED})
        with run.capture():
            # Per user: enroll on n0 typings, replay sessions 2-4 + impostor attempts, test on 5-8.
            t, models = E.adaptation(X, spec, n0=n0)
            t["model"] = fam
            m = t[["eer_static", "eer_adapted", "eer_oracle", "genuine_accepted", "impostors_absorbed"]].mean()
            print(m.round(4).to_dict())
        ADAPT.append(t)
        p = U.paired_test(t.eer_adapted, t.eer_static)
        mets = {**U.summarize(t.eer_adapted), "static_mean_eer": float(t.eer_static.mean()),
                "oracle_mean_eer": float(t.eer_oracle.mean()), "adapted_vs_static": p,
                "impostors_absorbed_total": int(t.impostors_absorbed.sum()),
                "impostor_attempts_total": int(t.impostor_attempts.sum()),
                "genuine_accept_rate": float(t.genuine_accepted.sum() / t.genuine_attempts.sum())}
        save_model(run, models, {"family": fam, "chosen": best, "n0": n0})
        save_run(run, t, mets, index={"key_settings": json.dumps({**best, "n0": n0})})
ADAPT = pd.concat(ADAPT, ignore_index=True)
adapt_table = ADAPT.groupby(["n0", "model"]).agg(
    eer_static=("eer_static", "mean"), eer_adapted=("eer_adapted", "mean"), eer_oracle=("eer_oracle", "mean"),
    genuine_accept_rate=("genuine_accepted", lambda v: v.sum() / (150 * len(v))),
    impostors_absorbed=("impostors_absorbed", "sum"), users_with_absorbed=("impostors_absorbed", lambda v: int((v > 0).sum()))
).reset_index()
adapt_table["p_adapted_vs_static"] = [U.paired_test(g.eer_adapted, g.eer_static)["wilcoxon_p"]
                                      for _, g in ADAPT.groupby(["n0", "model"])]
run = start_run("table_adaptation", {"model": "subset", "features": "log21", "protocol": "template adaptation",
                                     "split_short": "adapt->test", "n_train": [12, 24], "sampling": "first", "seed": SEED,
                                     "impostor_attempts_per_user": 50})
with run.capture():
    print(adapt_table.round(4).to_string())
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, n0 in zip(ax, (12, 24)):
    d = adapt_table[adapt_table.n0 == n0].set_index("model")[["eer_static", "eer_adapted", "eer_oracle"]]
    d.plot.bar(ax=a, rot=30)
    a.set_title(f"template adaptation, n0={n0}")
    a.set_ylabel("mean EER (sessions 5-8)")
fig.tight_layout()
save_figure(run, fig, "template_adaptation", report=True)
save_run(run, ADAPT, {"table": adapt_table}, tables={"adaptation_table": adapt_table})
display(adapt_table.round(4))

## 5. Speed normalization (first-n enrollment)

(a) 21 log features; (b) rhythm only: subtract each typing's mean log-time, drop one feature -> 20;
(c) (b) + mean log-time. Each Gaussian-family model is re-tuned on validation for each feature set.
The flow is omitted here to keep the run time reasonable.

In [ ]:
SPEED_FAMS = [f for f in E.FAMILIES if not f.startswith("RealNVP")]
SPEED = []
for fs in ("log21", "rhythm20", "rhythm20+mean"):
    Xf, _ = U.make_features(df, fs)
    run = start_run(f"speed_{fs}", {"model": SPEED_FAMS, "features": fs, "protocol": "official test, first-n enrollment",
                                    "split_short": "val->test", "selection": "re-tuned on validation per feature set",
                                    "n_train": list(E.ENROLL_N), "sampling": "first", "seed": SEED})
    chosen = {}
    with run.capture():
        for fam in SPEED_FAMS:
            # Re-tune on validation for this feature set, then first-n enrollment on test.
            best, table = E.tune_family(Xf, fam, verbose=False)
            chosen[fam] = best
            t = E.enrollment(Xf, E.final_spec(fam, best, table), samplings=("first",))
            SPEED.append(t.assign(model=fam, features=fs))
            print(f"{fs:14s} {fam:22s} {best}  EER@200 {t[t.n == 200].eer.mean():.4f}")
    run.config["key_settings"] = chosen
    d = pd.concat([s for s in SPEED if s.features.iloc[0] == fs])
    save_model(run, None, {"chosen_per_family": chosen})
    save_run(run, d, {"mean_eer_by_model_n": d.groupby(["model", "n"]).eer.mean().unstack()},
             index={"mean_eer": float(d[(d.n == 200) & (d.model == BASE)].eer.mean()), "model": "all (EER = baseline@200)"})
SPEED = pd.concat(SPEED, ignore_index=True)
speed_table = SPEED.groupby(["model", "features", "n"]).eer.mean().unstack("n")

# Paired tests of (b) rhythm20 and (c) rhythm20+mean against (a) log21, for each model,
# at a realistic one-sitting enrollment (n=24) and at full enrollment (n=200).
ptests = []
for fam in SPEED_FAMS:
    for n in (24, 200):
        a = SPEED[(SPEED.model == fam) & (SPEED.features == "log21") & (SPEED.n == n)].sort_values("user").eer
        for fs in ("rhythm20", "rhythm20+mean"):
            b = SPEED[(SPEED.model == fam) & (SPEED.features == fs) & (SPEED.n == n)].sort_values("user").eer
            p = U.paired_test(b.to_numpy(), a.to_numpy())
            ptests.append({"model": fam, "n": n, "features": fs, "eer": b.mean(), "eer_log21": a.mean(),
                           "diff": p["mean_diff"], "diff_ci": f"[{p['diff_ci95_low']:+.4f}, {p['diff_ci95_high']:+.4f}]",
                           "wilcoxon_p": p["wilcoxon_p"]})
speed_tests = pd.DataFrame(ptests)
run = start_run("table_speed", {"model": "Gaussian family + baselines", "features": ["log21", "rhythm20", "rhythm20+mean"],
                                "protocol": "first-n enrollment", "split_short": "test", "n_train": list(E.ENROLL_N),
                                "sampling": "first", "seed": SEED})
with run.capture():
    print(speed_table.round(4).to_string()); print(speed_tests.round(4).to_string())
fig, ax = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for a, fs in zip(ax, ("log21", "rhythm20", "rhythm20+mean")):
    for fam in SPEED_FAMS:
        m = SPEED[(SPEED.features == fs) & (SPEED.model == fam)].groupby("n").eer.mean()
        a.plot(m.index, m.values, marker="o", label=fam)
    a.set_xscale("log"); a.set_xticks(list(E.ENROLL_N), [str(n) for n in E.ENROLL_N])
    a.set_title(fs); a.set_xlabel("first n enrollment typings")
ax[0].set_ylabel("mean EER"); ax[-1].legend(fontsize=7)
fig.tight_layout()
save_figure(run, fig, "speed_normalization", report=True)
save_run(run, SPEED, {"table": speed_table.reset_index()}, tables={"speed_table": speed_table.reset_index(), "speed_paired_tests": speed_tests})
display(speed_tests.round(4))

## 6. GMM diagnostics: do components align with sessions?

In [ ]:
run = start_run("gmm_diagnostics", {"model": "GMM", "key_settings": TUNED["GMM"][0], "features": "log21",
                                    "protocol": "official (training typings 1-200)", "split_short": "train",
                                    "n_train": 200, "sampling": "first", "seed": SEED})
gmms = OFFICIAL["GMM"].models
with run.capture():
    ks = pd.Series({u: m.k_ for u, m in gmms.items()})
    print("K chosen by BIC:", ks.value_counts().sort_index().to_dict())
    gdiag, ctabs = E.gmm_session_alignment(X, gmms)
    if len(gdiag):
        print(gdiag.round(3).to_string())
        print("mean ARI vs session:", round(gdiag.ari_vs_session.mean(), 3),
              "| mean ARI vs early(1-2)/late(3-4):", round(gdiag.ari_vs_early_late.mean(), 3))
        for u in list(ctabs)[:5]:
            print(f"\nuser {u}\n", ctabs[u].to_string())
if len(gdiag):
    show = list(ctabs)[:6]
    fig, ax = plt.subplots(1, len(show) + 1, figsize=(3.2 * (len(show) + 1), 3.2))
    for a, u in zip(ax, show):
        ct = ctabs[u]
        a.imshow(ct.values, cmap="Blues", aspect="auto")
        a.set_xticks(range(ct.shape[1])); a.set_yticks(range(4), [f"s{s}" for s in ct.index])
        for (i, j), v in np.ndenumerate(ct.values):
            a.text(j, i, v, ha="center", va="center", fontsize=8)
        a.set_title(f"user {u} (ARI {gdiag.set_index('user').ari_vs_session[u]:.2f})", fontsize=9)
        a.set_xlabel("component"); a.grid(False)
    w = np.concatenate([np.sort(gmms[u].gmm_.weights_)[::-1][1:] for u in gdiag.user])
    ax[-1].hist(w, bins=15); ax[-1].set_title("weights of minor components"); ax[-1].set_xlabel("weight")
    fig.tight_layout()
    save_figure(run, fig, "gmm_components_vs_sessions", report=True)
save_run(run, gdiag if len(gdiag) else None, {"k_counts": ks.value_counts().to_dict(),
         "mean_ari_vs_session": float(gdiag.ari_vs_session.mean()) if len(gdiag) else None},
         index={"mean_eer": float(OFFICIAL["GMM"].eers.mean())})

## Summary

In [ ]:
print(f"total time {(time.time() - T0) / 60:.1f} min")
print("\nOFFICIAL\n", official_table[["model", "val_eer", "mean_eer", "sd", "ci95", "test_ll_median", "diff_vs_base", "diff_ci95", "wilcoxon_p"]].round(4).to_string())
print("\nOPEN-SET\n", openset_table[["model", "mean_eer", "official_mean_eer", "ci95", "diff_vs_base", "wilcoxon_p"]].round(4).to_string())
print("\nENROLLMENT (first n)\n", enroll_table.xs("first", level="sampling").round(4).to_string())
print("\nADAPTATION\n", adapt_table.round(4).to_string())